# 📖 Notebook 1: Threat Modeling with STRIDE

Before you write a single line of code, you should ask: **"What could go wrong?"**

That's what threat modeling is — a structured way to identify security risks in your system **before** attackers find them.

At Microsoft, threat modeling is **mandatory** for every product under the SDL (Security Development Lifecycle). It's done during the **design phase**, before implementation begins.

## Learning Objectives

By the end of this notebook, you'll understand:
- What threat modeling is and why every team should do it
- The STRIDE framework for categorizing threats
- How to draw a Data Flow Diagram (DFD) for your application
- How to identify attack surfaces
- How to build a threat model for our Flask web app

## 🛠️ Setup

Start the infrastructure first:

```bash
cd enterprise-patterns/security-review
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
# We'll use these to interact with our Flask app and database
import requests
import json

BASE_URL = "http://localhost:5001"

# Verify the Flask app is running
try:
    resp = requests.get(f"{BASE_URL}/health")
    print(f"✅ Flask app is running: {resp.json()}")
except requests.ConnectionError:
    print("❌ Flask app is not running. Run: docker-compose up -d")

## What is Threat Modeling?

**Threat modeling** is the process of identifying potential security threats to your system, then designing countermeasures.

Think of it like this: before building a house, an architect checks:
- Is the area prone to flooding? → Build on higher ground
- Are there earthquakes? → Reinforce the foundation
- Is the neighborhood safe? → Add locks and alarms

In software, we do the same thing:
- Can someone steal user credentials? → Add encryption + MFA
- Can someone inject malicious SQL? → Use parameterized queries
- Can someone access internal services? → Add network segmentation

### The Four Questions of Threat Modeling

Microsoft's SDL uses these four questions:

| # | Question | Example Answer |
|---|----------|----------------|
| 1 | What are we building? | A web app with user login, product search, comments |
| 2 | What can go wrong? | SQL injection, stolen passwords, XSS attacks |
| 3 | What are we doing about it? | Parameterized queries, bcrypt hashing, input escaping |
| 4 | Did we do a good job? | Security tests pass, pen test finds no critical issues |

## The STRIDE Framework

**STRIDE** is a mnemonic developed at Microsoft for categorizing security threats. Each letter represents a different type of attack:

| Letter | Threat | What It Means | Real-World Example |
|--------|--------|---------------|--------------------|
| **S** | Spoofing | Pretending to be someone else | Using stolen credentials to log in |
| **T** | Tampering | Modifying data you shouldn't | Changing the price of an item in a request |
| **R** | Repudiation | Denying you did something | Claiming you never placed that order |
| **I** | Information Disclosure | Exposing data that should be private | Leaking user emails through an API |
| **D** | Denial of Service | Making the system unavailable | Flooding the login endpoint with requests |
| **E** | Elevation of Privilege | Gaining access you shouldn't have | Regular user accessing admin endpoints |

### STRIDE Maps to Security Properties

Each STRIDE category is the **opposite** of a security property:

| Threat | Security Property |
|--------|-------------------|
| Spoofing | **Authentication** — verify identity |
| Tampering | **Integrity** — data hasn't been modified |
| Repudiation | **Non-repudiation** — actions are logged |
| Info Disclosure | **Confidentiality** — data is protected |
| Denial of Service | **Availability** — system stays up |
| Elevation of Privilege | **Authorization** — proper access control |

## Step 1: Draw the Data Flow Diagram (DFD)

The first step in threat modeling is understanding **how data flows** through your system.

A Data Flow Diagram uses four elements:

| Symbol | Element | Meaning |
|--------|---------|---------|
| Rectangle | External Entity | Users, third-party services (outside your control) |
| Circle | Process | Your code that transforms data |
| Parallel Lines | Data Store | Databases, file systems, caches |
| Arrow | Data Flow | Data moving between elements |

Here's the DFD for our Flask app:

```
┌──────────┐    HTTP requests    ┌──────────────────┐    SQL queries    ═══════════════
│  Browser │ ─────────────────▶ │   Flask App      │ ────────────────▶ ║  PostgreSQL  ║
│  (User)  │ ◀───────────────── │   (Port 5001)    │ ◀──────────────── ║  (Users,     ║
└──────────┘    HTML/JSON        │                  │    query results  ║  Products,   ║
                                 │  /api/login      │                   ║  Comments)   ║
                                 │  /api/products   │    cache ops      ═══════════════
                                 │  /api/transfer   │ ────────────────▶ ═══════════════
                                 │  /api/fetch-url  │ ◀──────────────── ║    Redis     ║
                                 │  /comments       │                   ║  (Sessions,  ║
                                 └──────────────────┘                   ║  Rate Limits)║
                                         │                              ═══════════════
                                         │ HTTP (SSRF)
                                         ▼
                                 ┌──────────────────┐
                                 │  External URLs   │
                                 │  (user-provided) │
                                 └──────────────────┘
```

### Trust Boundaries

**Trust boundaries** are lines where the level of trust changes. Data crossing a trust boundary needs extra validation.

In our app:
- **Browser → Flask**: User input is untrusted (biggest attack surface)
- **Flask → Postgres**: App-to-database is semi-trusted (but SQL injection can break this)
- **Flask → External URLs**: The app reaches out to untrusted URLs (SSRF risk)

## Step 2: Identify Attack Surfaces

The **attack surface** is every point where an attacker can interact with your system. Smaller is better.

Let's enumerate the attack surface of our Flask app:

In [ ]:
# Let's map out the attack surface of our Flask app
# In a real threat model, you'd use a spreadsheet or tool like Microsoft Threat Modeling Tool

attack_surface = [
    {
        "endpoint": "/api/login",
        "method": "POST",
        "input": "username, password (JSON body)",
        "trust_level": "Unauthenticated",
        "data_accessed": "User credentials in Postgres",
        "risks": ["Brute force", "Credential stuffing", "Information disclosure"],
    },
    {
        "endpoint": "/api/products/search",
        "method": "GET",
        "input": "q (query parameter)",
        "trust_level": "Unauthenticated",
        "data_accessed": "Product data in Postgres",
        "risks": ["SQL injection", "Data exfiltration"],
    },
    {
        "endpoint": "/comments/<product_id>",
        "method": "GET",
        "input": "product_id (URL path)",
        "trust_level": "Unauthenticated",
        "data_accessed": "Comments (user-generated content)",
        "risks": ["XSS via stored comments", "Path traversal"],
    },
    {
        "endpoint": "/api/comments",
        "method": "POST",
        "input": "user_id, product_id, content (JSON body)",
        "trust_level": "Unauthenticated",
        "data_accessed": "Writes to comments table",
        "risks": ["XSS payload injection", "Spam", "Missing auth"],
    },
    {
        "endpoint": "/api/transfer",
        "method": "POST",
        "input": "from_user, to_user, amount (JSON body)",
        "trust_level": "Unauthenticated",
        "data_accessed": "Financial operation",
        "risks": ["CSRF", "Missing authentication", "Missing authorization"],
    },
    {
        "endpoint": "/api/fetch-url",
        "method": "GET",
        "input": "url (query parameter)",
        "trust_level": "Unauthenticated",
        "data_accessed": "Any URL (internal or external)",
        "risks": ["SSRF", "Internal network scanning", "Data exfiltration"],
    },
]

print("🎯 Attack Surface Analysis")
print("=" * 80)
for entry in attack_surface:
    print(f"\n📍 {entry['method']} {entry['endpoint']}")
    print(f"   Input: {entry['input']}")
    print(f"   Trust: {entry['trust_level']}")
    print(f"   Data:  {entry['data_accessed']}")
    print(f"   Risks: {', '.join(entry['risks'])}")

## Step 3: Apply STRIDE to Each Component

Now we apply STRIDE to each part of the data flow diagram. For each component, we ask: **is this vulnerable to Spoofing? Tampering? Repudiation? Information Disclosure? Denial of Service? Elevation of Privilege?**

Let's build a threat model programmatically:

In [ ]:
# Building a threat model using STRIDE
# In practice, teams use tools like Microsoft Threat Modeling Tool or OWASP Threat Dragon

stride_categories = {
    "S": "Spoofing",
    "T": "Tampering",
    "R": "Repudiation",
    "I": "Information Disclosure",
    "D": "Denial of Service",
    "E": "Elevation of Privilege",
}

threat_model = [
    # --- Login Endpoint ---
    {
        "component": "/api/login",
        "stride": "S",
        "threat": "Attacker brute-forces passwords to impersonate a user",
        "severity": "High",
        "mitigation": "Rate limiting (Redis counter), account lockout after 5 attempts",
    },
    {
        "component": "/api/login",
        "stride": "I",
        "threat": "Error message reveals whether username exists",
        "severity": "Medium",
        "mitigation": "Return same error for wrong username AND wrong password",
    },
    {
        "component": "/api/login",
        "stride": "R",
        "threat": "No audit log of login attempts — can't trace breaches",
        "severity": "Medium",
        "mitigation": "Log all login attempts (success and failure) to audit_log table",
    },
    # --- Product Search ---
    {
        "component": "/api/products/search",
        "stride": "T",
        "threat": "SQL injection modifies query to extract/delete data",
        "severity": "Critical",
        "mitigation": "Parameterized queries (never concatenate user input into SQL)",
    },
    {
        "component": "/api/products/search",
        "stride": "I",
        "threat": "SQL injection extracts user passwords and emails",
        "severity": "Critical",
        "mitigation": "Parameterized queries + principle of least privilege on DB user",
    },
    # --- Comments ---
    {
        "component": "/comments/<product_id>",
        "stride": "T",
        "threat": "XSS payload in comment executes JavaScript in other users' browsers",
        "severity": "High",
        "mitigation": "HTML-escape all user content, Content-Security-Policy header",
    },
    # --- Transfer ---
    {
        "component": "/api/transfer",
        "stride": "S",
        "threat": "CSRF: malicious site triggers transfer on behalf of logged-in user",
        "severity": "Critical",
        "mitigation": "CSRF tokens, SameSite cookies, Origin header validation",
    },
    {
        "component": "/api/transfer",
        "stride": "E",
        "threat": "No authentication required — anyone can transfer funds",
        "severity": "Critical",
        "mitigation": "Require valid JWT token, verify user owns the source account",
    },
    # --- Fetch URL (SSRF) ---
    {
        "component": "/api/fetch-url",
        "stride": "I",
        "threat": "SSRF: attacker reads internal services (Redis, Postgres, metadata)",
        "severity": "Critical",
        "mitigation": "URL allowlist, block private IP ranges, HTTPS only",
    },
    {
        "component": "/api/fetch-url",
        "stride": "D",
        "threat": "Attacker makes server fetch huge files, exhausting memory",
        "severity": "Medium",
        "mitigation": "Response size limits, timeouts, rate limiting",
    },
]

print("🔍 STRIDE Threat Model for Security Demo App")
print("=" * 90)

for t in threat_model:
    category = stride_categories[t['stride']]
    print(f"\n🏷️  [{t['stride']}] {category} — {t['component']}")
    print(f"   Threat:      {t['threat']}")
    print(f"   Severity:    {t['severity']}")
    print(f"   Mitigation:  {t['mitigation']}")

## Step 4: Prioritize Threats

Not all threats are equal. We use a **risk matrix** to prioritize:

```
                    Impact
              Low    Medium    High
         ┌─────────┬─────────┬─────────┐
  High   │ Medium  │  High   │Critical │  ← Likelihood
         ├─────────┼─────────┼─────────┤
  Medium │  Low    │ Medium  │  High   │
         ├─────────┼─────────┼─────────┤
  Low    │  Low    │  Low    │ Medium  │
         └─────────┴─────────┴─────────┘
```

At Microsoft, the SDL requires that **all Critical and High severity threats must be mitigated before release**. Medium threats need a documented plan.

In [ ]:
# Let's count threats by severity and STRIDE category
from collections import Counter

severity_counts = Counter(t["severity"] for t in threat_model)
stride_counts = Counter(stride_categories[t["stride"]] for t in threat_model)

print("📊 Threat Summary")
print("=" * 40)
print("\nBy Severity:")
for severity in ["Critical", "High", "Medium", "Low"]:
    count = severity_counts.get(severity, 0)
    bar = "█" * count
    print(f"  {severity:10s} {bar} ({count})")

print("\nBy STRIDE Category:")
for letter, name in stride_categories.items():
    count = stride_counts.get(name, 0)
    bar = "█" * count
    print(f"  [{letter}] {name:25s} {bar} ({count})")

# Critical + High must be fixed before release (per Microsoft SDL)
must_fix = severity_counts.get("Critical", 0) + severity_counts.get("High", 0)
print(f"\n🚨 Must fix before release: {must_fix} threats")
print(f"📋 Need documented plan:   {severity_counts.get('Medium', 0)} threats")

## Step 5: Verify Threats with Real Attacks

Let's verify one of our identified threats is real. We said the login endpoint reveals whether a user exists. Let's test it:

In [ ]:
# Verify threat: Information Disclosure on /api/login
# The vulnerable endpoint returns different errors for "user not found" vs "wrong password"

print("Testing /api/login information disclosure...\n")

# Test with a username that EXISTS in the database
resp1 = requests.post(f"{BASE_URL}/api/login", json={
    "username": "alice",
    "password": "wrong-password"
})
print(f"Existing user, wrong password:")
print(f"  Status: {resp1.status_code}")
print(f"  Body:   {resp1.json()}")

# Test with a username that DOES NOT exist
resp2 = requests.post(f"{BASE_URL}/api/login", json={
    "username": "nonexistent_user_xyz",
    "password": "wrong-password"
})
print(f"\nNon-existent user:")
print(f"  Status: {resp2.status_code}")
print(f"  Body:   {resp2.json()}")

print("\n⚠️  Notice the difference!")
print("The vulnerable endpoint returns 404 for missing users and a different")
print("response for existing users. An attacker can use this to enumerate")
print("valid usernames before even trying passwords.")

# Now test the SAFE endpoint
print("\n" + "=" * 60)
print("Testing /api/login/safe (fixed version)...\n")

resp3 = requests.post(f"{BASE_URL}/api/login/safe", json={
    "username": "alice",
    "password": "wrong-password"
})
print(f"Existing user, wrong password:")
print(f"  Status: {resp3.status_code}")
print(f"  Body:   {resp3.json()}")

resp4 = requests.post(f"{BASE_URL}/api/login/safe", json={
    "username": "nonexistent_user_xyz",
    "password": "wrong-password"
})
print(f"\nNon-existent user:")
print(f"  Status: {resp4.status_code}")
print(f"  Body:   {resp4.json()}")

print("\n✅ The safe endpoint returns the same error in both cases!")
print("An attacker can't tell if a username exists or not.")

## Threat Model Document Template

At Microsoft and other large companies, threat models are documented formally. Here's a template:

```
# Threat Model: Security Demo Web Application

## 1. System Overview
- Flask web application serving product catalog
- PostgreSQL for data storage
- Redis for session management and rate limiting
- Exposed on port 5001

## 2. Trust Boundaries
- Internet → Flask App (untrusted)
- Flask App → PostgreSQL (semi-trusted)
- Flask App → Redis (semi-trusted)
- Flask App → External URLs (untrusted)

## 3. Data Classification
- HIGH: User passwords, JWT secrets, API keys
- MEDIUM: User email addresses, session tokens
- LOW: Product names, prices, public comments

## 4. STRIDE Analysis
- 5 Critical threats identified
- 3 High severity threats
- 3 Medium severity threats
- All Critical/High must be mitigated before launch

## 5. Mitigations
- [See threat table above]

## 6. Sign-off
- Security team review: PENDING
- Pen test: PENDING
- SDL compliance: PENDING
```

## Attack Surface Reduction

One of the most effective security measures is **reducing the attack surface** — removing features and endpoints that aren't needed.

### Checklist for Attack Surface Reduction

| Question | Action |
|----------|--------|
| Do all endpoints need to be public? | Add authentication where possible |
| Does the app need to fetch arbitrary URLs? | Remove or restrict `/api/fetch-url` |
| Does the app need debug mode in production? | Disable `debug=True` |
| Are all database columns necessary in API responses? | Return only needed fields |
| Are default credentials changed? | Always change defaults |
| Are unused ports closed? | Only expose what's needed |

In [ ]:
# Let's verify our app's attack surface by checking which endpoints exist

endpoints_to_check = [
    ("GET",  "/health"),
    ("GET",  "/api/products/search?q=laptop"),
    ("GET",  "/api/products/search/safe?q=laptop"),
    ("GET",  "/comments/1"),
    ("GET",  "/comments/1/safe"),
    ("POST", "/api/login"),
    ("POST", "/api/login/safe"),
    ("POST", "/api/transfer"),
    ("POST", "/api/transfer/safe"),
    ("GET",  "/api/fetch-url?url=http://httpbin.org/get"),
    ("GET",  "/api/fetch-url/safe?url=https://httpbin.org/get"),
    ("GET",  "/api/audit-log"),
]

print("🎯 Endpoint Accessibility Check")
print("=" * 70)
for method, path in endpoints_to_check:
    try:
        if method == "GET":
            resp = requests.get(f"{BASE_URL}{path}", timeout=5)
        else:
            resp = requests.post(f"{BASE_URL}{path}", json={}, timeout=5)
        status = "✅" if resp.status_code < 500 else "❌"
        print(f"  {status} {method:5s} {path:45s} → {resp.status_code}")
    except Exception as e:
        print(f"  ❌ {method:5s} {path:45s} → Error: {e}")

print("\n⚠️  Notice: ALL endpoints are accessible without authentication!")
print("In a real app, most of these should require a valid session or JWT token.")

## 🔑 Key Takeaways

1. **Threat model BEFORE you code** — it's cheaper to fix design issues than implementation bugs
2. **Use STRIDE** to systematically check for each category of threat
3. **Draw a Data Flow Diagram** to visualize where data goes and where trust boundaries are
4. **Minimize attack surface** — remove, restrict, or authenticate everything you can
5. **Document and prioritize** — Critical and High threats must be fixed before release
6. **Verify with real tests** — don't just assume mitigations work

## ➡️ Next: Notebook 2 — Common Vulnerabilities

Now that we've identified threats, let's **exploit them** and then **fix them**.